In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
import pandas as pd
from typing import List
from data_loader import fetch_categories_mmlu, load_mmlu_dataset

## Loading the MMLU dataset
normative_category = [
    "moral_disputes",
    "philosophy",
    "world_religions",
    "us_foreign_policy",
    "sociology",
    "professional_psychology",
    "professional_law",
    "moral_scenarios",
    "human_sexuality",
    "international_law",
]

control_category = ["college_mathematics",
    "college_physics",
    "formal_logic",
    "logical_fallacies",
    "college_computer_science",
]

dataset_subjects = normative_category + control_category

mmlu_full_df = fetch_categories_mmlu(dataset_subjects)

samples_per_subject = 50
samples_examples_per_subject = 5

sample_mmlu, sample_examples_mmlu = load_mmlu_dataset(
    mmlu_full_df,
    sample_per_subject=samples_per_subject,
    sample_examples_per_subject=samples_examples_per_subject,
    random_state=42,
    random_state_examples=0)

print(f"Sample MMLU shape: {sample_mmlu.shape}")
print(f"Sample Examples MMLU shape: {sample_examples_mmlu.shape}")
print("Subjects in sample_mmlu:", sample_mmlu["subject"].nunique())
print("Subjects in sample_examples_mmlu:", sample_examples_mmlu["subject"].nunique())



### Loading the full set of samples: 
sample_mmlu_full = pd.read_csv("data/samples_mmlu_full.csv", index_col=False)
print(f"\n\nFull sample MMLU shape: {sample_mmlu_full.shape}")
print("Subjects in sample_mmlu_full:", sample_mmlu_full["subject"].nunique())
print("List of subjects in sample_mmlu_full: ", sample_mmlu_full["subject"].unique())

Sample MMLU shape: (750, 7)
Sample Examples MMLU shape: (75, 7)
Subjects in sample_mmlu: 15
Subjects in sample_examples_mmlu: 15


Full sample MMLU shape: (750, 7)
Subjects in sample_mmlu_full: 15


In [3]:
from dotenv import load_dotenv
import openai
import os, torch, numpy as np

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_AZURE_OPENAI": "Azure OpenAI",
    "ENDPOINT_AZURE_OPENAI": "Azure OpenAI Endpoint",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")

client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

# Zero-shot and Few Shot on MMLU

In [4]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields


### Use this variable to test on 4 categories with more samples (from 9)
MMLU_DATA_FULL = True

ZERO_SHOT = False # Set to false for Few Shot
nb_examples_few_shots = 3
    
for nb_examples_few_shots in [1, 2, 4, 5]:
    case = mmlu_case
    case_name = case.case_name
    
    if MMLU_DATA_FULL:
        
        data = sample_mmlu_full
    
        if ZERO_SHOT:
    
            output_file = f"results/{model_filename}/zero_shot/classic/results_mmlu_zero_shot_full.csv"
            classifier = ZeroShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
            )
    
        else:
            output_file = f"results/{model_filename}/few_shot/classic/results_mmlu_few_shot_{nb_examples_few_shots}_shot_full.csv"
            classifier = FewShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
                n_shots=nb_examples_few_shots,
                examples_df=sample_examples_mmlu,
            )
    
    else:
        data = sample_mmlu
    
        if ZERO_SHOT:
            output_file = f"results/{model_filename}/zero_shot/classic/results_mmlu_zero_shot.csv"
            classifier = ZeroShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
            )
        else:
            output_file = f"results/{model_filename}/few_shot/classic/results_mmlu_few_shot_{nb_examples_few_shots}_shot.csv"
            classifier = FewShot(
                case=case,
                client=client,
                model=model,
                max_tokens=300,
                n_shots=nb_examples_few_shots,
                examples_df=sample_examples_mmlu,
            )
    
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    if os.path.exists(output_file):
        df_out_existing = pd.read_csv(output_file)
        done_ids = set(df_out_existing["sample_id"])
        rows = df_out_existing.to_dict(orient="records")
        print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
    else:
        done_ids = set()
        rows = []
    
    
    try:
        for idx, row in tqdm(data.iterrows(), total=len(data)):
            if idx in done_ids:
                continue
    
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()
    
            try:
                predicted_label, stats = classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            except RateLimitError as e:
                print(f"\nRate limit hit at sample {idx}. Saving progress.")
                break
            except Exception as e:
                print(f"\nError at sample {idx}: {e}. Skipping.")
                continue
    
            additional = get_additional_fields(row, case_name) 
    
            results = {
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": mapped_label,
                "max_tokens": classifier.max_tokens,
                "tokens_used": stats["tokens_used"],
                "prompt_tokens": stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "latency": stats["latency"],
                **additional,
            }
    
            rows.append(results)
    
    except KeyboardInterrupt:
        print("=== Interrupted manually. Saving progress...")
    
    finally:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"✅ Saved {len(df_out)} rows to {output_file}")
        
        y_true = df_out["true_label"].astype(int)
        y_pred = df_out["pred_label"].astype(int)
    
        print("=== Classification Report ===\n")
        print(classification_report(y_true, y_pred))
    
        print("\n=== Confusion Matrix ===\n")
        labels = sorted(set(y_true) | set(y_pred))
        conf_matrix = confusion_matrix(y_true, y_pred)
        print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
    
        accuracy = (y_true == y_pred).mean()
        print(f"\n=== Accuracy: {accuracy:.2%} ===")

100%|██████████| 579/579 [06:53<00:00,  1.40it/s]


✅ Saved 579 rows to results/openai_4.1_mini/few_shot/classic/results_mmlu_few_shot_1_shot_full.csv


ValueError: invalid literal for int() with base 10: 'II only'

## Role-playing in Zero-shot and Few-shots settings (3 examples)

In [7]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS


selected_profiles = [f"profile{i}" for i in range(1, 61)]

case = mmlu_case
case_name = "mmlu"
data = sample_mmlu
max_tokens = 300

role_playing = "passive"
person_set = PERSON_ETHNICS


for person_key in selected_profiles:

    output_file = f"results/{model_filename}/few_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_few_shot_3examples.csv"

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        df_out_existing = pd.read_csv(output_file)
        done_ids = set(df_out_existing["sample_id"])
        rows = df_out_existing.to_dict(orient="records")
        print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
    else:
        done_ids = set()
        rows = []
    
    zero_shot_classifier = FewShot(
        case=case,
        client=client,
        model=model,
        max_tokens=max_tokens,
        person_key=person_key,
        role_playing=role_playing,
        person_set=person_set,
        examples_df=sample_examples_mmlu
    )
    
    try:
        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"mmlu | {person_key} | few-shot | {role_playing}"):
            if idx in done_ids:
                continue
    
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()
    
            try:
                predicted_label, stats = zero_shot_classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            except RateLimitError as e:
                print(f"Error : {e}")
                print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                continue
            except Exception as e:
                print(f"ERROR at sample {idx}: {e}")
                continue
    
            additional = get_additional_fields(row, case_name) 
    
            results = {
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": mapped_label,
                "max_tokens": zero_shot_classifier.max_tokens,
                "tokens_used": stats["tokens_used"],
                "prompt_tokens": stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "latency": stats["latency"],
                **additional,
            }
    
            rows.append(results)
    
    except KeyboardInterrupt:
        print("=== Interrupted manually. Saving progress...")
    
    finally:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"✅ Saved {len(df_out)} rows to {output_file}")
        
        y_true = df_out["true_label"].astype(int)
        y_pred = df_out["pred_label"].astype(int)
    
        print("=== Classification Report ===\n")
        print(classification_report(y_true, y_pred))
    
        print("\n=== Confusion Matrix ===\n")
        labels = sorted(set(y_true) | set(y_pred))
        conf_matrix = confusion_matrix(y_true, y_pred)
        print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
    
        accuracy = (y_true == y_pred).mean()
        print(f"\n=== Accuracy for mmlu | {person_key} | {role_playing}: {accuracy:.2%} ===")

=== Resuming from last index... 750 samples already completed.


mmlu | profile1 | few-shot | passive: 100%|██████████| 750/750 [00:00<00:00, 71030.51it/s]


✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile1_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.77      0.76      0.76       181
           1       0.73      0.79      0.76       182
           2       0.80      0.76      0.78       188
           3       0.80      0.78      0.79       199

    accuracy                           0.77       750
   macro avg       0.77      0.77      0.77       750
weighted avg       0.77      0.77      0.77       750


=== Confusion Matrix ===

     0    1    2    3
0  137   22   12   10
1   17  144    9   12
2   14   14  143   17
3   11   18   15  155

=== Accuracy for mmlu | profile1 | passive: 77.20% ===
=== Resuming from last index... 750 samples already completed.


mmlu | profile2 | few-shot | passive: 100%|██████████| 750/750 [00:00<00:00, 75733.15it/s]


✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile2_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.75      0.74      0.74       181
           1       0.74      0.80      0.77       182
           2       0.80      0.78      0.79       188
           3       0.82      0.79      0.81       199

    accuracy                           0.78       750
   macro avg       0.78      0.78      0.78       750
weighted avg       0.78      0.78      0.78       750


=== Confusion Matrix ===

     0    1    2    3
0  134   23   14   10
1   18  145    8   11
2   15   13  146   14
3   12   14   15  158

=== Accuracy for mmlu | profile2 | passive: 77.73% ===
=== Resuming from last index... 750 samples already completed.


...

✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile58_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.76      0.77      0.76       181
           1       0.76      0.81      0.78       182
           2       0.79      0.76      0.77       188
           3       0.83      0.79      0.81       199

    accuracy                           0.78       750
   macro avg       0.78      0.78      0.78       750
weighted avg       0.78      0.78      0.78       750


=== Confusion Matrix ===

     0    1    2    3
0  139   21   11   10
1   16  148   11    7
2   16   14  142   16
3   12   13   16  158

=== Accuracy for mmlu | profile58 | passive: 78.27% ===
=== Resuming from last index... 750 samples already completed.


mmlu | profile59 | few-shot | passive: 100%|██████████| 750/750 [00:00<00:00, 76837.52it/s]


✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile59_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.75      0.76      0.76       181
           1       0.76      0.81      0.78       182
           2       0.80      0.77      0.78       188
           3       0.83      0.81      0.82       199

    accuracy                           0.79       750
   macro avg       0.79      0.79      0.79       750
weighted avg       0.79      0.79      0.79       750


=== Confusion Matrix ===

     0    1    2    3
0  138   21   12   10
1   17  147    9    9
2   15   14  144   15
3   13   11   14  161

=== Accuracy for mmlu | profile59 | passive: 78.67% ===
=== Resuming from last index... 750 samples already completed.


mmlu | profile60 | few-shot | passive: 100%|██████████| 750/750 [00:00<00:00, 77431.40it/s]


✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile60_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.76      0.76      0.76       181
           1       0.75      0.80      0.77       182
           2       0.78      0.76      0.77       188
           3       0.82      0.80      0.81       199

    accuracy                           0.78       750
   macro avg       0.78      0.78      0.78       750
weighted avg       0.78      0.78      0.78       750


=== Confusion Matrix ===

     0    1    2    3
0  137   20   14   10
1   17  145   11    9
2   14   15  143   16
3   12   13   15  159

=== Accuracy for mmlu | profile60 | passive: 77.87% ===


In [41]:
df_out = pd.read_csv("results/openai_4.1_mini/few_shot/classic/results_mmlu_few_shot_4_shot.csv")

y_true = df_out["true_label"].astype(int)
y_pred = df_out["pred_label"].astype(int)
print("=== Classification Report ===\n")
print(classification_report(y_true, y_pred))
print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true) | set(y_pred))
conf_matrix = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true == y_pred).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

df_out_norm = df_out[df_out["category"]=="normative"]

y_true_norm = df_out_norm["true_label"].astype(int)
y_pred_norm = df_out_norm["pred_label"].astype(int)
print("=== Classification Report ===\n")
print(classification_report(y_true_norm, y_pred_norm))
print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true_norm) | set(y_pred_norm))
conf_matrix = confusion_matrix(y_true_norm, y_pred_norm)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true_norm == y_pred_norm).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

df_out_control = df_out[df_out["category"] == "control"]

y_true_control = df_out_control["true_label"].astype(int)
y_pred_control = df_out_control["pred_label"].astype(int)

print("=== Classification Report ===\n")
print(classification_report(y_true_control, y_pred_control))

print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true_control) | set(y_pred_control))
conf_matrix = confusion_matrix(y_true_control, y_pred_control)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true_control == y_pred_control).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

=== Classification Report ===

              precision    recall  f1-score   support

           0       0.77      0.81      0.79       181
           1       0.78      0.82      0.80       182
           2       0.82      0.79      0.81       188
           3       0.84      0.79      0.82       199

    accuracy                           0.80       750
   macro avg       0.80      0.80      0.80       750
weighted avg       0.80      0.80      0.80       750


=== Confusion Matrix ===

     0    1    2    3
0  147   18    8    8
1   15  149    8   10
2   16   12  149   11
3   13   12   17  157

=== Accuracy: 80.27% ===
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.82      0.88      0.85       123
           1       0.84      0.90      0.87       127
           2       0.91      0.85      0.88       132
           3       0.90      0.84      0.87       118

    accuracy                           0.87       500
   macro avg  

# Roleplaying

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS

case = mmlu_case
case_name = "mmlu"
data = full_df_capped

# Done:
#  Zero shots profiles: 1 to 4, 6 to 14, 16 to 19, 21 to 29, 31 to 34, 41 to 60
#  Few shots profiles: 1 to 4, 51 to 54,

#next ones
a = [f"profile{i}" for i in range(1, 5)]
b = [f"profile{i}" for i in range(51, 55)]

c = [f"profile{i}" for i in range(56, 60)]
d = [f"profile{i}" for i in range(41, 45)]

selected_profiles = c+d   ####
role_playing = "passive"
person_set = PERSON_ETHNICS

USE_FEW_SHOT = True
USE_ZERO_SHOT = False

for person_key in selected_profiles:
    if USE_FEW_SHOT:
        output_file_prefix = f"few_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_few_shot_3examples"
    else:
        output_file_prefix = f"zero_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_zero_shot"

    df2 = pd.read_csv(f"results/{model_filename}/{output_file_prefix}.csv")
    
    subjects_to_keep = ["moral_scenarios", "professional_law", "college_mathematics", "formal_logic"]
    if "subject" in df2.columns and subjects_to_keep:
        df2_filtered = df2[df2["subject"].isin(subjects_to_keep)].reset_index(drop=True)
    else:
        df2_filtered = df2
    

    output_file = f"results/{model_filename}/{output_file_prefix}_full.csv"

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if os.path.exists(output_file):
        df_out_existing = pd.read_csv(output_file)
        done_ids = set(df_out_existing["sample_id"])
        rows = df_out_existing.to_dict(orient="records")
        print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
    else:
        done_ids = set()
        rows = []

    if USE_FEW_SHOT:
        zero_shot_classifier = FewShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
            n_shots=3,
            examples_df=full_examples_df,
            person_key=person_key,
            role_playing=role_playing,
            person_set=person_set
        )
        loop_desc = f"mmlu | {person_key} | few-shot | {role_playing}"
    elif USE_ZERO_SHOT:
        zero_shot_classifier = ZeroShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
            person_key=person_key,
            role_playing=role_playing,
            person_set=person_set
        )
        loop_desc = f"mmlu | {person_key} | zero-shot | {role_playing}"
    else:
        raise ValueError("Set USE_FEW_SHOT or USE_ZERO_SHOT to True.")

    try:
        for idx, row in tqdm(data.iterrows(), total=len(data), desc=loop_desc):
            if idx in done_ids:
                continue

            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            try:
                predicted_label, stats = zero_shot_classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            except RateLimitError as e:
                print(f"Error : {e}")
                print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                continue
            except Exception as e:
                print(f"ERROR at sample {idx}: {e}")
                continue

            additional = get_additional_fields(row, case_name)

            results = {
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": mapped_label,
                "max_tokens": zero_shot_classifier.max_tokens,
                "tokens_used": stats["tokens_used"],
                "prompt_tokens": stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "latency": stats["latency"],
                **additional,
            }

            rows.append(results)

    except KeyboardInterrupt:
        print("=== Interrupted manually. Saving progress...")

    finally:
        df_out_prev = pd.DataFrame(rows)
        df_out = pd.concat([df2_filtered, df_out_prev], ignore_index=True)

        desired_order = [
            "sample_id", "text", "true_label", "pred_label",
            "max_tokens", "tokens_used", "prompt_tokens", "completion_tokens", "latency",
            "subject", "category"
        ]

        final_cols = [c for c in desired_order if c in df_out.columns] + \
                     [c for c in df_out.columns if c not in desired_order]

        df_out = df_out[final_cols]

        df_out_filt = df_out[df_out["subject"].isin(subjects_to_keep)].reset_index(drop=True)
        df_out_filt["sample_id"] = df_out_filt.index

        print(df_out_filt.value_counts("subject"))

        df_out_filt.to_csv(output_file, index=False)
        print(f"✅ Saved {len(df_out_filt)} rows to {output_file}")

        y_true = df_out_filt["true_label"].astype(int)
        y_pred = df_out_filt["pred_label"].astype(int)

        print("=== Classification Report ===\n")
        print(classification_report(y_true, y_pred))

        print("\n=== Confusion Matrix ===\n")
        labels = sorted(set(y_true) | set(y_pred))
        conf_matrix = confusion_matrix(y_true, y_pred)
        print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

        accuracy = (y_true == y_pred).mean()
        print(f"\n=== Accuracy for {loop_desc}: {accuracy:.2%} ===")


mmlu | profile1 | few-shot | passive: 100%|██████████| 379/379 [04:01<00:00,  1.57it/s]


subject
moral_scenarios        180
professional_law       180
formal_logic           122
college_mathematics     97
Name: count, dtype: int64
✅ Saved 579 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile1_passive/results_mmlu_few_shot_3examples_full.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.68      0.64      0.66       151
           1       0.62      0.62      0.62       124
           2       0.66      0.58      0.62       138
           3       0.63      0.72      0.67       166

    accuracy                           0.64       579
   macro avg       0.64      0.64      0.64       579
weighted avg       0.65      0.64      0.64       579


=== Confusion Matrix ===

    0   1   2    3
0  96  23  16   16
1  17  77   9   21
2  15  10  80   33
3  14  15  17  120

=== Accuracy for mmlu | profile1 | few-shot | passive: 64.42% ===


mmlu | profile2 | few-shot | passive: 100%|██████████| 379/379 [03:45<00:00,  1.68it/s]


subject
moral_scenarios        180
professional_law       180
formal_logic           122
college_mathematics     97
Name: count, dtype: int64
✅ Saved 579 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile2_passive/results_mmlu_few_shot_3examples_full.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.67      0.66      0.66       151
           1       0.62      0.63      0.63       124
           2       0.66      0.57      0.61       138
           3       0.66      0.74      0.70       166

    accuracy                           0.66       579
   macro avg       0.65      0.65      0.65       579
weighted avg       0.66      0.66      0.65       579


=== Confusion Matrix ===

     0   1   2    3
0  100  24  13   14
1   18  78  10   18
2   18  11  79   30
3   14  12  17  123

=== Accuracy for mmlu | profile2 | few-shot | passive: 65.63% ===


mmlu | profile3 | few-shot | passive: 100%|██████████| 379/379 [04:08<00:00,  1.53it/s]


...

mmlu | profile54 | few-shot | passive: 100%|██████████| 379/379 [03:39<00:00,  1.72it/s]


subject
moral_scenarios        180
professional_law       180
formal_logic           122
college_mathematics     97
Name: count, dtype: int64
✅ Saved 579 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile54_passive/results_mmlu_few_shot_3examples_full.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.65      0.62      0.63       151
           1       0.62      0.64      0.63       124
           2       0.69      0.61      0.65       138
           3       0.66      0.75      0.70       166

    accuracy                           0.66       579
   macro avg       0.66      0.65      0.65       579
weighted avg       0.66      0.66      0.66       579


=== Confusion Matrix ===

    0   1   2    3
0  93  26  15   17
1  19  79   8   18
2  17   8  84   29
3  13  14  15  124

=== Accuracy for mmlu | profile54 | few-shot | passive: 65.63% ===


mmlu | profile56 | few-shot | passive:   4%|▎         | 14/379 [00:08<03:43,  1.63it/s]


=== Interrupted manually. Saving progress...
subject
college_mathematics    64
formal_logic           50
moral_scenarios        50
professional_law       50
Name: count, dtype: int64
✅ Saved 214 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile56_passive/results_mmlu_few_shot_3examples_full.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.63      0.49      0.55        53
           1       0.58      0.67      0.62        49
           2       0.67      0.67      0.67        46
           3       0.66      0.70      0.68        66

    accuracy                           0.64       214
   macro avg       0.64      0.63      0.63       214
weighted avg       0.64      0.64      0.63       214


=== Confusion Matrix ===

    0   1   2   3
0  26  15   6   6
1   7  33   0   9
2   3   3  31   9
3   5   6   9  46

=== Accuracy for mmlu | profile56 | few-shot | passive: 63.55% ===


mmlu | profile57 | few-shot | passive:   5%|▌         | 20/379 [00:11<03:17,  1.82it/s]